# Velocity Motors Dataset - Exploratory Data Analysis

Analyze data quality characteristics of the Velocity Motors dataset to identify gaps and inform generator improvements.

**Tables Analyzed**: 12 tables across 3 domains (Sales, CRM, Operations)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Configure display
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", None)
sns.set_theme(style="whitegrid")

# Data path
DATA_PATH = Path("../dataset_generators/data/velocity_motors")

In [ ]:
# Define all tables to load
TABLE_NAMES = [
    "salespersons",
    "vehicles",
    "orders",
    "order_items",
    "customer_segments",
    "customers",
    "interactions",
    "leads",
    "warehouse_locations",
    "suppliers",
    "parts_inventory",
    "service_orders",
]

# Load all parquet files into a dictionary
tables = {}

if not DATA_PATH.exists():
    print(f"ERROR: Data directory not found at {DATA_PATH.resolve()}")
    print("Please generate the dataset first using:")
    print("  uv run python dataset_generators/generate_velocity_motors.py --scale 0.1")
else:
    for table_name in TABLE_NAMES:
        file_path = DATA_PATH / f"{table_name}.parquet"
        if file_path.exists():
            tables[table_name] = pd.read_parquet(file_path)
            print(f"Loaded {table_name}: {len(tables[table_name]):,} rows")
        else:
            print(f"WARNING: {table_name}.parquet not found")

print(f"\nTotal tables loaded: {len(tables)}/12")

## 1. Data Overview

Summary statistics for each table including row counts, column counts, and memory usage.

In [ ]:
# Build overview table
overview_data = []

for table_name, df in tables.items():
    memory_mb = df.memory_usage(deep=True).sum() / (1024 * 1024)
    overview_data.append(
        {
            "Table": table_name,
            "Rows": len(df),
            "Columns": len(df.columns),
            "Memory (MB)": round(memory_mb, 2),
            "Dtypes": ", ".join(df.dtypes.astype(str).unique()),
        }
    )

overview_df = pd.DataFrame(overview_data)
overview_df = overview_df.sort_values("Rows", ascending=False).reset_index(drop=True)

# Display with formatting
print("=" * 80)
print("VELOCITY MOTORS DATASET OVERVIEW")
print("=" * 80)
print(f"\nTotal rows: {overview_df['Rows'].sum():,}")
print(f"Total memory: {overview_df['Memory (MB)'].sum():.2f} MB")
print()
display(overview_df.style.format({"Rows": "{:,}", "Memory (MB)": "{:.2f}"}))

## 2. NULL Analysis

Identify columns with missing values. Distinguish between:
- **Semantic NULLs**: Intentionally null (e.g., `company_name` for Individual customers)
- **Missing NULLs**: Data quality issues (e.g., `notes` fields always null)

In [ ]:
# Calculate NULL rates per column per table
null_analysis = []

for table_name, df in tables.items():
    for col in df.columns:
        null_count = df[col].isna().sum()
        null_rate = null_count / len(df) * 100 if len(df) > 0 else 0
        if null_count > 0:
            null_analysis.append(
                {
                    "Table": table_name,
                    "Column": col,
                    "NULL Count": null_count,
                    "NULL Rate (%)": round(null_rate, 2),
                    "Total Rows": len(df),
                }
            )

null_df = pd.DataFrame(null_analysis)
if not null_df.empty:
    null_df = null_df.sort_values("NULL Rate (%)", ascending=False).reset_index(drop=True)
    print(f"Found {len(null_df)} columns with NULL values:\n")
    display(null_df)
else:
    print("No NULL values found in any table.")

In [ ]:
# Create NULL rate heatmap
if not null_df.empty:
    # Pivot for heatmap
    null_pivot = null_df.pivot_table(index="Table", columns="Column", values="NULL Rate (%)", fill_value=0)

    # Filter to columns with > 0% NULLs
    cols_with_nulls = null_pivot.columns[null_pivot.sum() > 0]
    null_pivot_filtered = null_pivot[cols_with_nulls]

    if not null_pivot_filtered.empty:
        fig, ax = plt.subplots(figsize=(14, 8))
        sns.heatmap(null_pivot_filtered, annot=True, fmt=".1f", cmap="YlOrRd", cbar_kws={"label": "NULL Rate (%)"})
        plt.title("NULL Rate (%) by Table and Column", fontsize=14)
        plt.xlabel("Column")
        plt.ylabel("Table")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()
    else:
        print("No columns with significant NULL rates to visualize.")
else:
    print("No NULL values to visualize.")

In [ ]:
# Document semantic vs missing NULLs
print("=" * 60)
print("NULL CLASSIFICATION")
print("=" * 60)

semantic_nulls = {
    ("customers", "company_name"): "NULL for Individual segment customers (70% of data)",
    ("leads", "customer_id"): "NULL for unconverted leads (~80%)",
    ("leads", "converted_date"): "NULL for unconverted leads (~80%)",
    ("service_orders", "customer_rating"): "NULL when customer hasn't rated (~40%)",
}

missing_nulls = {
    ("interactions", "notes"): "ALWAYS NULL - placeholder not implemented",
    ("service_orders", "notes"): "ALWAYS NULL - placeholder not implemented",
}

print("\n[SEMANTIC NULLs - Expected by Design]")
for (table, col), reason in semantic_nulls.items():
    if table in tables and col in tables[table].columns:
        rate = tables[table][col].isna().mean() * 100
        print(f"  - {table}.{col}: {rate:.1f}% NULL - {reason}")

print("\n[MISSING NULLs - Data Quality Gaps]")
for (table, col), reason in missing_nulls.items():
    if table in tables and col in tables[table].columns:
        rate = tables[table][col].isna().mean() * 100
        print(f"  - {table}.{col}: {rate:.1f}% NULL - {reason}")

## 3. Distribution Analysis

Analyze the distribution of key categorical and numeric fields.

In [ ]:
# Categorical distributions
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Categorical Variable Distributions", fontsize=14, y=1.02)

# 1. Customer segments
if "customers" in tables and "customer_segments" in tables:
    merged = tables["customers"].merge(
        tables["customer_segments"][["segment_id", "segment_name"]], on="segment_id", how="left"
    )
    segment_counts = merged["segment_name"].value_counts()
    axes[0, 0].pie(segment_counts, labels=segment_counts.index, autopct="%1.1f%%", startangle=90)
    axes[0, 0].set_title("Customer Segment Distribution")

# 2. Order status
if "orders" in tables:
    status_counts = tables["orders"]["status"].value_counts()
    axes[0, 1].barh(status_counts.index, status_counts.values, color="steelblue")
    axes[0, 1].set_xlabel("Count")
    axes[0, 1].set_title("Order Status Distribution")

# 3. Vehicle makes
if "vehicles" in tables:
    make_counts = tables["vehicles"]["make"].value_counts()
    axes[0, 2].barh(make_counts.index, make_counts.values, color="forestgreen")
    axes[0, 2].set_xlabel("Count")
    axes[0, 2].set_title("Vehicle Make Distribution")

# 4. Lead source
if "leads" in tables:
    source_counts = tables["leads"]["source"].value_counts()
    axes[1, 0].barh(source_counts.index, source_counts.values, color="darkorange")
    axes[1, 0].set_xlabel("Count")
    axes[1, 0].set_title("Lead Source Distribution")

# 5. Service types
if "service_orders" in tables:
    service_counts = tables["service_orders"]["service_type"].value_counts()
    axes[1, 1].barh(service_counts.index, service_counts.values, color="purple")
    axes[1, 1].set_xlabel("Count")
    axes[1, 1].set_title("Service Type Distribution")

# 6. Salesperson region
if "salespersons" in tables:
    region_counts = tables["salespersons"]["region"].value_counts()
    axes[1, 2].barh(region_counts.index, region_counts.values, color="crimson")
    axes[1, 2].set_xlabel("Count")
    axes[1, 2].set_title("Salesperson Region Distribution")

plt.tight_layout()
plt.show()

In [ ]:
# Numeric distributions
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("Numeric Variable Distributions", fontsize=14, y=1.02)

# 1. Vehicle MSRP
if "vehicles" in tables:
    axes[0, 0].hist(tables["vehicles"]["msrp"], bins=50, color="steelblue", edgecolor="white")
    axes[0, 0].set_xlabel("MSRP ($)")
    axes[0, 0].set_ylabel("Count")
    axes[0, 0].set_title("Vehicle MSRP Distribution")
    axes[0, 0].ticklabel_format(style="plain", axis="x")

# 2. Customer lifetime value
if "customers" in tables:
    axes[0, 1].hist(tables["customers"]["lifetime_value"], bins=50, color="forestgreen", edgecolor="white")
    axes[0, 1].set_xlabel("Lifetime Value ($)")
    axes[0, 1].set_ylabel("Count")
    axes[0, 1].set_title("Customer Lifetime Value Distribution")
    axes[0, 1].ticklabel_format(style="plain", axis="x")

# 3. Customer credit score
if "customers" in tables:
    axes[0, 2].hist(tables["customers"]["credit_score"], bins=30, color="darkorange", edgecolor="white")
    axes[0, 2].set_xlabel("Credit Score")
    axes[0, 2].set_ylabel("Count")
    axes[0, 2].set_title("Customer Credit Score Distribution")

# 4. Service order total cost
if "service_orders" in tables:
    axes[1, 0].hist(tables["service_orders"]["total_cost"], bins=50, color="purple", edgecolor="white")
    axes[1, 0].set_xlabel("Total Cost ($)")
    axes[1, 0].set_ylabel("Count")
    axes[1, 0].set_title("Service Order Cost Distribution")

# 5. Interaction duration
if "interactions" in tables:
    axes[1, 1].hist(tables["interactions"]["duration_minutes"], bins=30, color="crimson", edgecolor="white")
    axes[1, 1].set_xlabel("Duration (minutes)")
    axes[1, 1].set_ylabel("Count")
    axes[1, 1].set_title("Interaction Duration Distribution")

# 6. Vehicle mileage
if "vehicles" in tables:
    axes[1, 2].hist(tables["vehicles"]["mileage"], bins=50, color="teal", edgecolor="white")
    axes[1, 2].set_xlabel("Mileage")
    axes[1, 2].set_ylabel("Count")
    axes[1, 2].set_title("Vehicle Mileage Distribution")

plt.tight_layout()
plt.show()

In [ ]:
# Date distribution: Order volume over time
if "orders" in tables:
    orders_df = tables["orders"].copy()
    orders_df["order_date"] = pd.to_datetime(orders_df["order_date"])
    orders_df["order_month"] = orders_df["order_date"].dt.to_period("M")

    monthly_orders = orders_df.groupby("order_month").size()

    fig, ax = plt.subplots(figsize=(14, 5))
    monthly_orders.plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
    ax.set_xlabel("Month")
    ax.set_ylabel("Order Count")
    ax.set_title("Monthly Order Volume (Expected: Q4 Spike in Nov-Dec)")

    # Rotate x-axis labels for readability
    plt.xticks(rotation=45, ha="right")

    # Highlight Q4 months (Oct, Nov, Dec)
    for i, period in enumerate(monthly_orders.index):
        if period.month in [10, 11, 12]:
            ax.patches[i].set_color("darkorange")

    plt.tight_layout()
    plt.show()

    # Calculate seasonality factor
    avg_monthly = monthly_orders.mean()
    q4_avg = monthly_orders[monthly_orders.index.month.isin([10, 11, 12])].mean()
    print(f"\nAverage monthly orders: {avg_monthly:,.0f}")
    print(f"Q4 average (Oct-Dec): {q4_avg:,.0f}")
    print(f"Q4 seasonality factor: {q4_avg / avg_monthly:.2f}x baseline")

## 4. Referential Integrity

Validate foreign key relationships and identify orphan references.

In [ ]:
# Define all foreign key relationships
fk_relationships = [
    # (child_table, child_fk_column, parent_table, parent_pk_column)
    ("customers", "segment_id", "customer_segments", "segment_id"),
    ("orders", "customer_id", "customers", "customer_id"),
    ("orders", "vehicle_id", "vehicles", "vehicle_id"),
    ("orders", "salesperson_id", "salespersons", "salesperson_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("interactions", "customer_id", "customers", "customer_id"),
    ("leads", "customer_id", "customers", "customer_id"),  # NULL for unconverted
    ("leads", "salesperson_id", "salespersons", "salesperson_id"),
    ("parts_inventory", "supplier_id", "suppliers", "supplier_id"),
    ("parts_inventory", "warehouse_id", "warehouse_locations", "warehouse_id"),
    ("service_orders", "customer_id", "customers", "customer_id"),
    ("service_orders", "vehicle_id", "vehicles", "vehicle_id"),
]

print("=" * 80)
print("FOREIGN KEY RELATIONSHIP VALIDATION")
print("=" * 80)

In [ ]:
# Check each FK relationship for orphans
fk_results = []

for child_table, child_fk, parent_table, parent_pk in fk_relationships:
    if child_table not in tables or parent_table not in tables:
        continue

    child_df = tables[child_table]
    parent_df = tables[parent_table]

    # Get non-null FK values
    child_fk_values = child_df[child_fk].dropna()
    parent_pk_values = set(parent_df[parent_pk])

    # Count orphans (FK values not in parent PK)
    orphans = ~child_fk_values.isin(parent_pk_values)
    orphan_count = orphans.sum()
    total_non_null = len(child_fk_values)
    orphan_rate = (orphan_count / total_non_null * 100) if total_non_null > 0 else 0

    # Also count NULLs
    null_count = child_df[child_fk].isna().sum()
    null_rate = null_count / len(child_df) * 100 if len(child_df) > 0 else 0

    status = "PASS" if orphan_count == 0 else "FAIL"

    fk_results.append(
        {
            "Child Table": child_table,
            "FK Column": child_fk,
            "Parent Table": parent_table,
            "Total Rows": len(child_df),
            "NULL FKs": null_count,
            "NULL Rate (%)": round(null_rate, 1),
            "Orphan Count": orphan_count,
            "Orphan Rate (%)": round(orphan_rate, 1),
            "Status": status,
        }
    )

fk_results_df = pd.DataFrame(fk_results)
display(fk_results_df.style.applymap(lambda x: "background-color: #ffcccc" if x == "FAIL" else "", subset=["Status"]))

In [ ]:
# Visualize orphan rates
fk_with_orphans = fk_results_df[fk_results_df["Orphan Rate (%)"] > 0].copy()

if not fk_with_orphans.empty:
    fk_with_orphans["Relationship"] = fk_with_orphans["Child Table"] + "." + fk_with_orphans["FK Column"]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(fk_with_orphans["Relationship"], fk_with_orphans["Orphan Rate (%)"], color="crimson")
    ax.set_xlabel("Orphan Rate (%)")
    ax.set_title("Foreign Key Relationships with Orphan References")

    # Add value labels
    for bar, val in zip(bars, fk_with_orphans["Orphan Rate (%)"]):
        ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2, f"{val:.1f}%", va="center")

    plt.tight_layout()
    plt.show()
else:
    print("No orphan FK references found.")

## 5. Duplicate Detection

Check for duplicate primary keys and near-duplicate records.

In [ ]:
# Define primary keys for each table
pk_mapping = {
    "salespersons": "salesperson_id",
    "vehicles": "vehicle_id",
    "orders": "order_id",
    "order_items": "order_item_id",
    "customer_segments": "segment_id",
    "customers": "customer_id",
    "interactions": "interaction_id",
    "leads": "lead_id",
    "warehouse_locations": "warehouse_id",
    "suppliers": "supplier_id",
    "parts_inventory": "part_id",
    "service_orders": "service_order_id",
}

print("=" * 60)
print("PRIMARY KEY UNIQUENESS CHECK")
print("=" * 60)

pk_results = []
for table_name, pk_col in pk_mapping.items():
    if table_name not in tables:
        continue

    df = tables[table_name]
    total = len(df)
    unique = df[pk_col].nunique()
    duplicates = total - unique
    status = "PASS" if duplicates == 0 else "FAIL"

    pk_results.append(
        {
            "Table": table_name,
            "PK Column": pk_col,
            "Total Rows": total,
            "Unique Values": unique,
            "Duplicates": duplicates,
            "Status": status,
        }
    )

pk_results_df = pd.DataFrame(pk_results)
display(
    pk_results_df.style.applymap(
        lambda x: "background-color: #ffcccc" if x == "FAIL" else "background-color: #ccffcc" if x == "PASS" else "",
        subset=["Status"],
    )
)

In [ ]:
# Near-duplicate detection for customers (similar names/emails)
if "customers" in tables:
    customers_df = tables["customers"].copy()

    print("\n" + "=" * 60)
    print("NEAR-DUPLICATE DETECTION: CUSTOMERS")
    print("=" * 60)

    # Check for duplicate emails
    email_counts = customers_df["email"].value_counts()
    duplicate_emails = email_counts[email_counts > 1]

    if not duplicate_emails.empty:
        print(f"\nFound {len(duplicate_emails)} duplicate email addresses:")
        print(duplicate_emails.head(10))
    else:
        print("\nNo duplicate email addresses found.")

    # Check for duplicate phone numbers
    phone_counts = customers_df["phone"].value_counts()
    duplicate_phones = phone_counts[phone_counts > 1]

    if not duplicate_phones.empty:
        print(f"\nFound {len(duplicate_phones)} duplicate phone numbers:")
        print(duplicate_phones.head(10))
    else:
        print("\nNo duplicate phone numbers found.")

    # Check for similar names (exact match on first + last)
    name_counts = customers_df.groupby(["first_name", "last_name"]).size().reset_index(name="count")
    duplicate_names = name_counts[name_counts["count"] > 1]

    if not duplicate_names.empty:
        print(f"\nFound {len(duplicate_names)} duplicate name combinations:")
        display(duplicate_names.sort_values("count", ascending=False).head(10))
    else:
        print("\nNo duplicate name combinations found.")

## 6. Outlier Detection

Identify statistical outliers and business logic violations.

In [ ]:
# Define numeric columns to check for outliers
numeric_checks = [
    ("vehicles", "msrp", "Vehicle MSRP"),
    ("vehicles", "mileage", "Vehicle Mileage"),
    ("customers", "lifetime_value", "Customer LTV"),
    ("customers", "credit_score", "Credit Score"),
    ("service_orders", "total_cost", "Service Cost"),
    ("service_orders", "duration_minutes", "Service Duration"),
    ("interactions", "duration_minutes", "Interaction Duration"),
    ("salespersons", "quota", "Sales Quota"),
]

print("=" * 70)
print("STATISTICAL OUTLIER DETECTION (IQR Method)")
print("=" * 70)
print("\nOutliers defined as values outside [Q1 - 1.5*IQR, Q3 + 1.5*IQR]\n")

outlier_results = []

for table_name, col, description in numeric_checks:
    if table_name not in tables or col not in tables[table_name].columns:
        continue

    data = tables[table_name][col].dropna()

    if len(data) == 0:
        continue

    q1 = data.quantile(0.25)
    q3 = data.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = data[(data < lower_bound) | (data > upper_bound)]
    outlier_rate = len(outliers) / len(data) * 100

    outlier_results.append(
        {
            "Field": f"{table_name}.{col}",
            "Description": description,
            "Min": data.min(),
            "Q1": q1,
            "Median": data.median(),
            "Q3": q3,
            "Max": data.max(),
            "Outlier Count": len(outliers),
            "Outlier Rate (%)": round(outlier_rate, 2),
        }
    )

outlier_df = pd.DataFrame(outlier_results)
display(outlier_df)

In [ ]:
# Business logic checks
print("\n" + "=" * 70)
print("BUSINESS LOGIC VALIDATION")
print("=" * 70)

logic_issues = []

# Check 1: Negative values where not expected
negative_checks = [
    ("vehicles", "msrp", "MSRP should be positive"),
    ("vehicles", "mileage", "Mileage should be non-negative"),
    ("customers", "lifetime_value", "LTV should be non-negative"),
    ("service_orders", "total_cost", "Cost should be non-negative"),
    ("service_orders", "labor_cost", "Labor cost should be non-negative"),
    ("service_orders", "parts_cost", "Parts cost should be non-negative"),
]

for table_name, col, rule in negative_checks:
    if table_name not in tables or col not in tables[table_name].columns:
        continue
    data = tables[table_name][col].dropna()
    violations = (data < 0).sum()
    if violations > 0:
        logic_issues.append({"Table": table_name, "Column": col, "Rule": rule, "Violations": violations})

# Check 2: Credit score out of valid range (300-850)
if "customers" in tables:
    credit_scores = tables["customers"]["credit_score"].dropna()
    invalid_scores = ((credit_scores < 300) | (credit_scores > 850)).sum()
    if invalid_scores > 0:
        logic_issues.append(
            {
                "Table": "customers",
                "Column": "credit_score",
                "Rule": "Credit score should be 300-850",
                "Violations": invalid_scores,
            }
        )

# Check 3: Future dates where not expected
from datetime import datetime

today = datetime.now()

date_checks = [
    ("customers", "customer_since", "Customer since date should be in the past"),
    ("salespersons", "hire_date", "Hire date should be in the past"),
]

for table_name, col, rule in date_checks:
    if table_name not in tables or col not in tables[table_name].columns:
        continue
    dates = pd.to_datetime(tables[table_name][col]).dropna()
    future_dates = (dates > today).sum()
    if future_dates > 0:
        logic_issues.append({"Table": table_name, "Column": col, "Rule": rule, "Violations": future_dates})

# Check 4: service_orders total_cost should equal labor_cost + parts_cost
if "service_orders" in tables:
    so = tables["service_orders"]
    expected_total = so["labor_cost"] + so["parts_cost"]
    cost_mismatch = (~np.isclose(so["total_cost"], expected_total, rtol=0.01)).sum()
    if cost_mismatch > 0:
        logic_issues.append(
            {
                "Table": "service_orders",
                "Column": "total_cost",
                "Rule": "total_cost should equal labor_cost + parts_cost",
                "Violations": cost_mismatch,
            }
        )

if logic_issues:
    logic_df = pd.DataFrame(logic_issues)
    display(logic_df)
else:
    print("\nNo business logic violations detected.")

## 7. Summary and Recommendations

Overall data quality assessment and gap summary.

In [ ]:
print("=" * 80)
print("VELOCITY MOTORS DATASET - DATA QUALITY SUMMARY")
print("=" * 80)

# Calculate quality metrics
total_tables = len(tables)
total_rows = sum(len(df) for df in tables.values())

# FK integrity score
if not fk_results_df.empty:
    fk_pass_rate = (fk_results_df["Status"] == "PASS").mean() * 100
else:
    fk_pass_rate = 100

# PK uniqueness score
if not pk_results_df.empty:
    pk_pass_rate = (pk_results_df["Status"] == "PASS").mean() * 100
else:
    pk_pass_rate = 100

# Null rate (excluding semantic nulls)
missing_null_tables = ["interactions", "service_orders"]
missing_null_cols = ["notes"]
null_issues = 0
for table in missing_null_tables:
    if table in tables:
        for col in missing_null_cols:
            if col in tables[table].columns:
                if tables[table][col].isna().all():
                    null_issues += 1

print("\nDataset Statistics:")
print(f"  - Tables: {total_tables}")
print(f"  - Total Rows: {total_rows:,}")

print("\nData Quality Scores:")
print(f"  - Primary Key Uniqueness: {pk_pass_rate:.0f}%")
print(f"  - Foreign Key Integrity: {fk_pass_rate:.0f}%")
print(f"  - Null Field Issues: {null_issues} columns always NULL")

In [ ]:
# Gap summary table
gaps = [
    {
        "Category": "Referential Integrity",
        "Issue": "orders.customer_id orphan FKs",
        "Severity": "HIGH",
        "Description": "~90% of orders reference non-existent customers at scale=0.1",
        "Recommendation": "Fix generator to ensure orders only reference existing customers",
    },
    {
        "Category": "Missing Data",
        "Issue": "interactions.notes always NULL",
        "Severity": "MEDIUM",
        "Description": "Notes field is never populated (placeholder)",
        "Recommendation": "Generate realistic interaction notes using templates",
    },
    {
        "Category": "Missing Data",
        "Issue": "service_orders.notes always NULL",
        "Severity": "MEDIUM",
        "Description": "Service notes field is never populated (placeholder)",
        "Recommendation": "Generate realistic service notes using templates",
    },
]

# Add FK issues found
if not fk_results_df.empty:
    for _, row in fk_results_df[fk_results_df["Status"] == "FAIL"].iterrows():
        if row["Orphan Rate (%)"] > 10:  # Only significant issues
            gaps.append(
                {
                    "Category": "Referential Integrity",
                    "Issue": f"{row['Child Table']}.{row['FK Column']} orphan FKs",
                    "Severity": "HIGH" if row["Orphan Rate (%)"] > 50 else "MEDIUM",
                    "Description": f"{row['Orphan Rate (%)']:.1f}% orphan rate ({row['Orphan Count']:,} records)",
                    "Recommendation": f"Ensure {row['FK Column']} values exist in {row['Parent Table']}",
                }
            )

# Remove duplicates (the predefined ones)
seen = set()
unique_gaps = []
for gap in gaps:
    key = gap["Issue"]
    if key not in seen:
        seen.add(key)
        unique_gaps.append(gap)

gaps_df = pd.DataFrame(unique_gaps)

print("\n" + "=" * 80)
print("DATA QUALITY GAPS AND RECOMMENDATIONS")
print("=" * 80 + "\n")

display(
    gaps_df.style.applymap(
        lambda x: "background-color: #ffcccc" if x == "HIGH" else "background-color: #ffffcc" if x == "MEDIUM" else "",
        subset=["Severity"],
    )
)

In [ ]:
# Final summary metrics
print("\n" + "=" * 80)
print("CLEANLINESS SCORE ESTIMATION")
print("=" * 80)

# Calculate weighted cleanliness score
scores = {
    "PK Uniqueness": (pk_pass_rate, 0.20),
    "FK Integrity": (fk_pass_rate, 0.30),
    "Completeness (non-semantic)": (100 - (null_issues * 10), 0.25),  # Penalty for null placeholder columns
    "Business Logic": (100 if not logic_issues else max(0, 100 - len(logic_issues) * 20), 0.25),
}

weighted_score = sum(score * weight for score, weight in scores.values())

print("\nComponent Scores:")
for name, (score, weight) in scores.items():
    print(f"  - {name}: {score:.1f}% (weight: {weight * 100:.0f}%)")

print(f"\n{'=' * 40}")
print(f"OVERALL CLEANLINESS SCORE: {weighted_score:.1f}%")
print(f"{'=' * 40}")

if weighted_score >= 90:
    grade = "EXCELLENT"
elif weighted_score >= 75:
    grade = "GOOD"
elif weighted_score >= 60:
    grade = "FAIR"
else:
    grade = "NEEDS IMPROVEMENT"

print(f"Grade: {grade}")
print("\nNote: Primary gap is FK integrity in orders.customer_id relationship.")
print("This is a known issue at scale=0.1 and should be investigated in the generator.")